# DimRed API Demo

This notebook demonstrates the complete workflow for using the DimRed API:

1. Create a project
2. Create a dataset
3. Add data points to the dataset
4. Create a prompt
5. Create a metric
6. Run prompt tuning
7. Poll for tuning session completion

## Setup

First, import the DimRed API client and configure logging.

In [ ]:
import json
import logging
import os
import sys

# Add the parent directory (dimred-examples) to Python path for importing client
# This handles both running from notebooks and via nbconvert
current_dir = os.getcwd()
dimred_root = current_dir

# Find the dimred-examples directory
if 'dimred-examples' in current_dir:
    # Split path and find dimred-examples root
    parts = current_dir.split('/')
    idx = next(i for i, p in enumerate(parts) if 'dimred-examples' in p)
    dimred_root = '/'.join(parts[:idx+1])

sys.path.insert(0, dimred_root)

from client import DimRedAPIClient

# Configure logging to output to stdout for better visibility in notebooks
logging.basicConfig(
    level=logging.INFO,
    format='[%(asctime)s] %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
    stream=sys.stdout,  # Output to stdout instead of stderr
    force=True  # Override any existing configuration
)
logger = logging.getLogger(__name__)

## Configuration

Set your API key and base URL here.

In [ ]:
# Configuration
import os
API_KEY = os.environ.get("DIMRED_API_KEY")
BASE_URL = "https://www.dimred.com"  # Updated URL

# Path to example data file
current_dir = os.getcwd()
dimred_root = current_dir

# Find the dimred-examples directory
if 'dimred-examples' in current_dir:
    # Split path and find dimred-examples root
    parts = current_dir.split('/')
    idx = next(i for i, p in enumerate(parts) if 'dimred-examples' in p)
    dimred_root = '/'.join(parts[:idx+1])

DATA_FILE_PATH = os.path.join(dimred_root, 'data', 'example.json')

# Initialize client
client = DimRedAPIClient(API_KEY, BASE_URL)

## Step 1: Create Project

In [3]:
project_id = client.create_project(
    project_name="Jupyter Demo Project",
    project_description="Testing DimRed API with Jupyter notebook"
)

print(f"✓ Created project: {project_id}")

[2025-10-21 23:58:31] INFO - Creating project: Jupyter Demo Project
[2025-10-21 23:58:32] INFO - ✓ Project created: 97e05fe6-c66d-4089-9903-f62e03a46640
✓ Created project: 97e05fe6-c66d-4089-9903-f62e03a46640


## Step 2: Create Dataset

In [4]:
dataset_id = client.create_dataset(
    project_id=project_id,
    dataset_name="Financial Crime Detection Dataset"
)

print(f"✓ Created dataset: {dataset_id}")

[2025-10-21 23:58:32] INFO - Creating dataset: Financial Crime Detection Dataset for project 97e05fe6-c66d-4089-9903-f62e03a46640
[2025-10-21 23:58:33] INFO - ✓ Dataset created: ds-ac43eb59-a710-4054-a12e-80355cd28f5c
✓ Created dataset: ds-ac43eb59-a710-4054-a12e-80355cd28f5c


## Step 3: Add Datapoints

Load data from the example file and add it to the dataset.

In [5]:
# Load data from file
with open(DATA_FILE_PATH, 'r') as f:
    example_data = json.load(f)

print(f"Loaded {len(example_data)} datapoints from {DATA_FILE_PATH}")

# Convert to API format
datapoints = []
for item in example_data:
    datapoints.append({
        "input_data": json.dumps(item["input"]),
        "expected_output": json.dumps(item["expected"])
    })

# Add to dataset
count = client.add_datapoints(dataset_id, datapoints)
print(f"✓ Added {count} datapoints")

Loaded 10 datapoints from /Users/jonathanjohannemann/Downloads/example_data (8).json
[2025-10-21 23:58:33] INFO - Adding 10 datapoints to dataset ds-ac43eb59-a710-4054-a12e-80355cd28f5c
[2025-10-21 23:58:34] INFO - ✓ Added 10 datapoints
✓ Added 10 datapoints


## Step 4: Create Prompt

Create a prompt for financial crime perpetrator detection.

In [ ]:
# Create prompt text
prompt_text = (
    "You are an expert financial crime analyst. Your task is to analyze news article "
    "snippets and determine whether the person mentioned is a perpetrator of financial crime.\n\n"
    "A person is a PERPETRATOR if:\n"
    "- They are explicitly charged, indicted, arrested, or accused of financial crimes\n"
    "- There is clear evidence of illegal activity (e.g., court documents, bank records)\n"
    "- They are directly involved in illegal financial transactions\n\n"
    "A person is NOT a perpetrator if:\n"
    "- They are law enforcement, prosecutors, or investigators\n"
    "- They are witnesses, victims, or observers\n"
    "- There is only speculation or suspicion without charges\n"
    "- They are community leaders or officials responding to crimes\n\n"
    "Respond with JSON containing:\n"
    "- is_perpetrator: true or false\n"
    "- reasoning: brief explanation of your decision"
)

# Create output schema for structured JSON response
output_schema = {
    "type": "object",
    "properties": {
        "is_perpetrator": {
            "type": "boolean",
            "description": "Whether the person is a perpetrator of financial crime"
        },
        "reasoning": {
            "type": "string",
            "description": "Brief explanation of the decision"
        }
    },
    "required": ["is_perpetrator", "reasoning"]
}

# Use the new API format
prompt_id = client.create_prompt(
    project_id=project_id,
    prompt_text=prompt_text,
    prompt_message_type="system",
    name="Financial Crime Perpetrator Detection",
    output_schema=output_schema
)

print(f"✓ Created prompt: {prompt_id}")

## Step 5: Create Metric

Create a metric to evaluate perpetrator classification accuracy.

In [7]:
metric_code = '''
import json

def metric_func(output, expected):
    """
    Check if the LLM correctly identified whether someone is a perpetrator.
    Returns 1.0 for correct classification, 0.0 for incorrect.
    """
    # Parse output and expected if they're strings
    if isinstance(output, str):
        try:
            output = json.loads(output)
        except json.JSONDecodeError:
            return 0.0

    if isinstance(expected, str):
        try:
            expected = json.loads(expected)
        except json.JSONDecodeError:
            return 0.0

    # Extract is_perpetrator field
    output_value = output.get("is_perpetrator")
    expected_value = expected.get("is_perpetrator")

    # Both must be present and match
    if output_value is None or expected_value is None:
        return 0.0

    # Return 1.0 if they match, 0.0 if they don't
    return 1.0 if output_value == expected_value else 0.0
'''

metric_id = client.create_metric(
    project_id=project_id,
    code=metric_code,
    metric_name="Perpetrator Classification Accuracy",
    metric_description="Measures whether the model correctly identifies perpetrators vs non-perpetrators"
)

print(f"✓ Created metric: {metric_id}")

[2025-10-21 23:58:35] INFO - Creating metric for project 97e05fe6-c66d-4089-9903-f62e03a46640
[2025-10-21 23:58:36] INFO - ✓ Metric created: 0897da43-f4d4-4ac8-a8b3-6528a9de55c7
✓ Created metric: 0897da43-f4d4-4ac8-a8b3-6528a9de55c7


## Step 6: Run Tuning

Start a prompt tuning session.

In [ ]:
# Run tuning using the new workflow API
workflow_response = client.run_workflow(
    project_id=project_id,
    dataset_id=dataset_id,
    prompt_id=prompt_id,
    metric_id=metric_id,
    model_name="gpt-4o-mini",  # Updated model name
    provider="openai",
    mode="tune",
    num_iterations=1
)

workflow_id = workflow_response.get("id") or workflow_response.get("tuning_session_id")

print(f"✓ Tuning started")
print(f"  Workflow ID: {workflow_id}")
print(f"  Status: {workflow_response.get('status')}")

## Step 7: Wait for Completion

Poll the tuning session until it completes. 

**Note:** This can take 10-30 minutes. Enable DEBUG logging below to see polling progress every 15 seconds.

In [9]:
# Optional: Enable DEBUG logging to see polling progress every 15 seconds
# Uncomment the line below if you want verbose output:
logging.getLogger().setLevel(logging.DEBUG)

In [ ]:
# Wait for workflow completion using new monitoring
final_result = client.wait_for_workflow_completion(
    workflow_id=workflow_id,
    poll_interval=10,
    timeout=3600
)

print("\n=== Final Results ===")
print(f"Workflow ID: {workflow_id}")
print(f"Status: {final_result.get('status')}")

# Extract metrics data
metrics_data = final_result.get("metrics", {})
final_metrics = metrics_data.get("final_metrics", {})
iteration_results = metrics_data.get("iteration_results", [])

if final_metrics:
    print("\nFinal Metrics:")
    for metric_name, value in final_metrics.items():
        print(f"  {metric_name}: {value}")

if iteration_results:
    print(f"\nCompleted {len(iteration_results)} iterations:")
    for result in iteration_results:
        iter_num = result.get("iteration", "?")
        iter_metrics = result.get("metrics", {})
        print(f"  Iteration {iter_num}: {iter_metrics}")

print(f"\nBest Prompt ID: {metrics_data.get('best_prompt_id', 'N/A')}")
print(f"Final Prompt ID: {metrics_data.get('final_prompt_id', 'N/A')}")

## Step 8: Fetch Best Prompt

Retrieve the best performing prompt from the tuning session.

In [ ]:
# Fetch and display the best prompt from tuning results
best_prompt_id = metrics_data.get('best_prompt_id')
final_prompt_id = metrics_data.get('final_prompt_id')

print("\n=== Best Prompt Information ===")

if best_prompt_id:
    print(f"Best Prompt ID: {best_prompt_id}")
    
    if final_prompt_id and final_prompt_id != best_prompt_id:
        print(f"Final Prompt ID: {final_prompt_id}")
    
    # Always fetch and display the best prompt
    print(f"\nFetching prompt details for ID: {best_prompt_id}")
    try:
        best_prompt = client.get_prompt(best_prompt_id)
        
        # Check if we got valid prompt data
        if best_prompt and isinstance(best_prompt, dict):
            if best_prompt.get('prompt_text'):
                print("\n✓ Successfully retrieved prompt")
                print(f"\nPrompt Text:")
                print("-" * 50)
                print(best_prompt['prompt_text'])
                print("-" * 50)
                
                if best_prompt.get('output_schema'):
                    print(f"\nOutput Schema:")
                    print(json.dumps(best_prompt['output_schema'], indent=2))
                    
                if best_prompt.get('prompt_message_type'):
                    print(f"\nMessage Type: {best_prompt['prompt_message_type']}")
            else:
                # Empty response, fetch the original prompt as fallback
                print(f"API response empty, fetching original prompt ID: {prompt_id}")
                original_prompt = client.get_prompt(prompt_id)
                if original_prompt and original_prompt.get('prompt_text'):
                    print("\n✓ Retrieved original prompt")
                    print(f"\nPrompt Text:")
                    print("-" * 50)
                    print(original_prompt['prompt_text'])
                    print("-" * 50)
                else:
                    print("✓ Prompt IDs captured successfully")
        else:
            print("✓ Prompt IDs captured successfully")
            
    except Exception as e:
        print(f"Could not fetch full prompt details: {e}")
        print("✓ Prompt IDs captured successfully")
    
    # Indicate whether optimization occurred
    if prompt_id == best_prompt_id:
        print("\n📊 Result: The original prompt performed best on this dataset")
    else:
        print("\n📊 Result: An optimized prompt was generated through tuning")
    
    print("\n📝 Use this prompt ID in future workflows:")
    print(f"   client.run_workflow(..., prompt_id='{best_prompt_id}', ...)")
    
else:
    print("Error: Best prompt ID not available in results")
    print("Check that the tuning workflow completed successfully.")